# 6. Test split: ground truth and scores (CPU, high RAM)

Same code as `03_score`, for the six test blocks. Run after `05_test_predict_gpu` has finished. Results go into
the same run (`RUN_TAG`) as the other blocks, so everything lives together; `07_test_report` shows the test scenes
on their own. Optional: `06_test_score_second_server` on a second server at the same time.

**Next:** `07_test_report`.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"   # where predictions, ground truth and results live.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
RUN_TAG = "phase7_front_medium"       # the folder of this run in persistent storage. A name from the time of the work:
                                      # every notebook of the run must use the same one, the results live under it
CAMERA = "front_medium"
MODELS = ["vggt_omega_512", "vggt_1b"]
BLOCKS = [("test", 0), ("test", 1), ("test", 2), ("test", 3), ("test", 11), ("test", 12)]
# The six test blocks that can be downloaded at the pinned dataset revision: 107 scenes. Nothing was tuned on them.
LIDAR_POLICY = "ouster_only"          # same six sensors in every scene, so ground-truth density is comparable
VALIDATE_DOWNLOAD = True              # run the dataset toolkit's own validator on each new block
REVERSE = False                       # forwards here; `06_test_score_second_server` walks the same list backwards,
                                      # so two CPU servers can share the work. This notebook alone does everything.
WORKERS = None                        # scenes scored at once. None = one per CPU core (max 8). The numbers do not depend on it

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

# build_cpp=True compiles the C++ geometry core on this server (about 15 s). Ground truth is then built
# with it, which gives exactly the same result as the Python reference, faster. If the build fails,
# everything still runs, in Python.
session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False, build_cpp=True)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)
$ cmake -S /content/vggt-omega-aura-benchmark/cpp -B /content/vggt-omega-aura-benchmark/cpp/build -DCMAKE_BUILD_TYPE=Release -Dpybind11_DIR=/usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11 -DPython_EXECUTABLE=/usr/bin/python3
$ cmake --build /content/vggt-omega-aura-benchmark/cpp/build --config Release -j
C++ core    : built


In [4]:
# --- 4. Process the blocks ---
import pandas as pd
from vggt_aura import aura_data as ad, pipeline as pl

pd.set_option("display.width", 220)
chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")   # faulty scenes the maintainers exclude
print("scenes excluded by the dataset:", len(EXCLUDED))
available = ad.available_blocks(chunks, scene_blocks, hub_files, [pl.CAMERA_LAYER, pl.LIDAR_LAYER])
import uuid
ME = ("backward-" if REVERSE else "forward-") + uuid.uuid4().hex[:6]      # this server's name on its claims
summaries, left_to_the_other = [], []
for split, block in (list(reversed(BLOCKS)) if REVERSE else list(BLOCKS)):
    if not pl.block_is_done(session.persist_root, RUN_TAG, MODELS, split, block) \
            and not pl.claim_block(session.persist_root, RUN_TAG, split, block, ME, max_age_s=1500):
        print(f"=== {pl.block_tag(split, block)}: the other server is on it, skipped ===")
        left_to_the_other.append((split, block))
        continue
    assert (split, block) in available.index, f"block {(split, block)} is not downloadable with camera + LiDAR"
    print(f"=== {pl.block_tag(split, block)} ({available.loc[(split, block), 'total_gb']} GB) ===")
    try:
        summary = pl.process_block(session, split, block, ad.block_scene_ids(scene_blocks, split, block, EXCLUDED), CAMERA, MODELS,
                                   RUN_TAG, lidar_policy=LIDAR_POLICY, validate=VALIDATE_DOWNLOAD,
                                   scene_names=ad.block_scene_names(scene_blocks, split, block, EXCLUDED), workers=WORKERS)
    finally:                 # a claim must not outlive a crash: a re-run gets a new name and would wait for it
        pl.release_claim(session.persist_root, RUN_TAG, split, block, ME)
    print(" ", summary)
    summaries.append(summary)
print()
print(pd.DataFrame([{k: v for k, v in s.items() if k not in ("sensors", "validation")} for s in summaries]).to_string(index=False))
still_open = [b for b in left_to_the_other if not pl.block_is_done(session.persist_root, RUN_TAG, MODELS, *b)]
if still_open:
    print()
    print("left to the other server and not finished yet:", still_open, "| if that server stopped, run this notebook again")

scenes excluded by the dataset: 8
=== test_block000000 (16.48 GB) ===
  downloading with the toolkit, decompressing with xz on all 8 cores


  fast unpack: {'archives': 5, 'xz_decompressed_on_all_cores': 3, 'download_s': 57.1, 'verify_and_decompress_s': 245.6, 'extract_s': 107.3}
  predictions: 0 made now, the rest loaded (1 s) | scoring 20 scenes with 8 worker(s)
  2026-05-28-15-28-52|75      68.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-04-14-05-43|45      65.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-15-28-52|83      65.5 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-15-28-52|81      48.3 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-15-28-52|96      67.5 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-15-28-52|69      54.5 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-09-57-03|108     48.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-09-57-03|106     48.0 s | vggt_omega_512: truth

  fast unpack: {'archives': 4, 'xz_decompressed_on_all_cores': 2, 'download_s': 40.4, 'verify_and_decompress_s': 211.2, 'extract_s': 98.3}
  predictions: 0 made now, the rest loaded (1 s) | scoring 20 scenes with 8 worker(s)
  2026-06-03-10-44-05|97      53.3 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|105     54.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|104     55.0 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-10-44-05|114     61.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|112     50.0 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-10-44-05|108     54.8 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|107     55.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-10-44-05|99      55.1 s | vggt_omega_512: truth 

In [5]:
# --- 5. What the run holds so far ---
for model in MODELS:
    rows, scenes = pl.load_run(session.persist_root, RUN_TAG, model)
    print(f"{model}: {scenes['scene_id'].nunique() if len(scenes) else 0} scenes in "
          f"{scenes[['split', 'block']].drop_duplicates().shape[0] if len(scenes) else 0} blocks")

vggt_omega_512: 414 scenes in 22 blocks
vggt_1b: 414 scenes in 22 blocks
